In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import polars as pl

from memory_profiler import profile

pd.set_option('display.max_columns', None)

In [ ]:
# @profile
def process_pandas(path: str):
    """
    Reads a path for a CSV file with pandas and return a DataFrame object after executing some processing.
    """
    df = pd.read_csv(path)
    grouped_df = (
        df.query("passenger_count > 1")
        .groupby(by="PULocationID")
        .agg({"fare_amount": "mean"})
        .sort_values(by="fare_amount", ascending=False)
        .reset_index()
    )

    return grouped_df

In [ ]:
# @profile
def process_polars(path: str):
    """
    Reads a path for a CSV file with polars and return a DataFrame object after executing some processing.
    """

    df = pl.scan_csv(path)
    df = df.filter(pl.col("passenger_count") > 1)
    grouped_df = (
        df.group_by("PULocationID")
        .agg(pl.col("fare_amount").mean())
        .sort(by="fare_amount", descending=True)
        .collect()
    )

    return grouped_df

In [ ]:
path = 'data/test_data.csv'

In [ ]:
%%time
df = process_pandas(path=path)
display(df)

In [ ]:
%%time
df = process_polars(path=path)
display(df)

"Ao reescrevermos nossa ingestão de dados, saímos de um modelo que carregava todo o histórico na memória (Pandas) para um modelo inteligente que só extrai o necessário (Polars). Isso reduz a necessidade de máquinas caras na nuvem (redução de custo de infraestrutura) e diminui o tempo de processamento de horas para minutos, permitindo que a área de negócios tenha análises atualizadas muito mais rápido."